<a href="https://colab.research.google.com/github/EnasIbrahimAli2005/Cats-and-Dogs-Classification-Project/blob/main/Cat_Dogs_classification_(Enas_Ibrahim).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d unmoved/30k-cats-and-dogs-150x150-greyscale

Dataset URL: https://www.kaggle.com/datasets/unmoved/30k-cats-and-dogs-150x150-greyscale
License(s): CC0-1.0
 97% 129M/133M [00:01<00:00, 119MB/s] 
100% 133M/133M [00:01<00:00, 81.0MB/s]


In [ ]:
!unzip 30k-cats-and-dogs-150x150-greyscale.zip

Streaming output truncated to the last 5000 lines.
  inflating: Animal Images/dogs/dog.441.jpg  
  inflating: Animal Images/dogs/dog.4410.jpg  
  inflating: Animal Images/dogs/dog.4411.jpg  
  inflating: Animal Images/dogs/dog.4412.jpg  
  inflating: Animal Images/dogs/dog.4413.jpg  
  inflating: Animal Images/dogs/dog.4414.jpg  
  inflating: Animal Images/dogs/dog.4415.jpg  
  inflating: Animal Images/dogs/dog.4416.jpg  
  inflating: Animal Images/dogs/dog.4417.jpg  
  inflating: Animal Images/dogs/dog.4418.jpg  
  inflating: Animal Images/dogs/dog.4419.jpg  
  inflating: Animal Images/dogs/dog.442.jpg  
  inflating: Animal Images/dogs/dog.4420.jpg  
  inflating: Animal Images/dogs/dog.4421.jpg  
  inflating: Animal Images/dogs/dog.4422.jpg  
  inflating: Animal Images/dogs/dog.4423.jpg  
  inflating: Animal Images/dogs/dog.4424.jpg  
  inflating: Animal Images/dogs/dog.4425.jpg  
  inflating: Animal Images/dogs/dog.4426.jpg  
  inflating: Animal Images/dogs/dog.4427.jpg  
  inflating

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the directory containing the 'cats' and 'dogs' folders
dataset_dir = '/content/Animal Images'

# Create an instance of ImageDataGenerator and rescale the images
datagen = ImageDataGenerator(rescale=1./255)  # Normalize pixel values (0-255 -> 0-1)

# Load the data from directory, assuming folder structure: /dataset/cats/ and /dataset/dogs/
train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=(150, 150),  # Resize all images to 150x150
    batch_size=32,           # Define batch size
    class_mode='binary',     # For binary classification (cats vs dogs)
    color_mode='grayscale',  # Use grayscale images
    shuffle=True             # Shuffle the data
)

Found 30061 images belonging to 2 classes.


In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import tensorflow as tf

# Paths to the dataset
dataset_dir = '/content/Animal Images'

# Initialize lists for images and labels
images = []
labels = []

# Iterate through the directory, assuming folder names 'cats' and 'dogs'
for label, category in enumerate(['cats', 'dogs']):
    folder_path = os.path.join(dataset_dir, category)
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read as grayscale
        img = cv2.resize(img, (150, 150))  # Resize to 150x150
        images.append(img)
        labels.append(label)  # 0 for cats, 1 for dogs

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)



# Reshape images to add the channel dimension (necessary for CNN)
images = images.reshape(-1, 150, 150, 1)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

# Build the CNN model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.60))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Binary classification
learning_rate = 0.001
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
# Compile the model
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/15
602/602 [==============================] - 1140s 2s/step - loss: 5.2380 - accuracy: 0.5048 - val_loss: 0.6917 - val_accuracy: 0.5351
Epoch 2/15
602/602 [==============================] - 1106s 2s/step - loss: 0.6912 - accuracy: 0.5073 - val_loss: 0.6917 - val_accuracy: 0.4900
Epoch 3/15
602/602 [==============================] - 1127s 2s/step - loss: 0.6898 - accuracy: 0.5138 - val_loss: 0.6918 - val_accuracy: 0.5399
Epoch 4/15
602/602 [==============================] - 1122s 2s/step - loss: 0.6891 - accuracy: 0.5109 - val_loss: 0.6975 - val_accuracy: 0.5324
Epoch 5/15
520/602 [========================>.....] - ETA: 2:20 - loss: 0.6887 - accuracy: 0.5168

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the test set
y_pred_prob = model.predict(X_test)

# Convert probabilities to binary labels (threshold at 0.5)
y_pred = np.where(y_pred_prob > 0.5, 1, 0)

# Print the classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))

188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Classification Report:

              precision    recall  f1-score   support

         Cat       0.60      0.29      0.39      3042
         Dog       0.52      0.81      0.64      2971

    accuracy                           0.54      6013
   macro avg       0.56      0.55      0.51      6013
weighted avg       0.56      0.54      0.51      6013

